# TP 8 — Sous-réseau et passerelle : décider avant d'envoyer

**Public cible :** étudiant, technicien ou NetOps débutant qui connaît déjà les adresses IPv4.  
**Durée :** 60 minutes. **Mode :** entièrement hors ligne, synthétique et déterministe.

Question fil rouge : pour joindre une destination, la machine doit-elle résoudre la destination elle-même sur le lien local, ou remettre le paquet à un prochain saut ?

Ce TP transforme les mots masque, sous-réseau, route et passerelle en décisions vérifiables. Il n'envoie aucun paquet et n'interroge aucune interface réelle.

## Contrat d'apprentissage et modèle mental

À la fin du TP, vous saurez :

1. relier **/24** à **255.255.255.0** et calculer l'adresse réseau par un **AND bit à bit** ;
2. identifier adresse réseau, broadcast et plage d'hôtes, avec les cas particuliers **/31** et **/32** ;
3. expliquer une décision locale ou distante par la **table de routage** et le **préfixe le plus long** ;
4. vérifier qu'une **default gateway** est on-link, puis nommer l'adresse réellement résolue par ARP ou ND ;
5. distinguer paquet IP de bout en bout et trame Ethernet par saut ;
6. transposer le raisonnement à IPv6 : /64, RA, ::/0, ND et absence de broadcast ;
7. séparer les rôles de passerelle, routeur, NAT, DNS et pare-feu.

Modèle compact : **destination IP → meilleure route → prochain saut → voisin L2 → trame**. Une passerelle par défaut n'est utilisée que si aucune route plus précise ne gagne.

In [ ]:
import base64
import hashlib
import ipaddress
import json

SEED = 2408

class RichHTML:
    def __init__(self, markup):
        self.markup = markup

    def _repr_html_(self):
        return self.markup

    def __repr__(self):
        return "RichHTML(deterministic)"

def html_table(title, columns, rows):
    head = "".join(
        f"<th style='padding:8px 10px;text-align:left;border-bottom:1px solid #334155'>{c}</th>"
        for c in columns
    )
    body = "".join(
        "<tr>" + "".join(
            f"<td style='padding:8px 10px;border-bottom:1px solid #1e293b'>{value}</td>"
            for value in row
        ) + "</tr>"
        for row in rows
    )
    return RichHTML(
        "<div style='font:14px system-ui;color:#e2e8f0;background:#0f172a;"
        "border:1px solid #334155;border-radius:12px;padding:14px;overflow:auto'>"
        f"<h3 style='margin:0 0 10px;color:#7dd3fc'>{title}</h3>"
        f"<table style='border-collapse:collapse;width:100%'><thead><tr>{head}</tr></thead>"
        f"<tbody>{body}</tbody></table></div>"
    )

def subnet_fact(cidr):
    interface = ipaddress.ip_interface(cidr)
    network = interface.network
    prefix = network.prefixlen
    and_result = ipaddress.ip_address(int(interface.ip) & int(network.netmask))
    if prefix <= 30:
        usable = f"{network.network_address + 1} → {network.broadcast_address - 1}"
        exception = "réseau et broadcast non attribuables"
    elif prefix == 31:
        usable = f"{network.network_address} ↔ {network.broadcast_address}"
        exception = "/31 : deux extrémités point-à-point, pas de broadcast usuel"
    else:
        usable = str(network.network_address)
        exception = "/32 : une seule adresse, route d'hôte"
    return {
        "cidr": cidr,
        "ip": str(interface.ip),
        "prefix": prefix,
        "mask": str(network.netmask),
        "and_result": str(and_result),
        "network": str(network.network_address),
        "broadcast": str(network.broadcast_address),
        "usable": usable,
        "exception": exception,
    }

fact_24 = subnet_fact("192.168.10.42/24")
assert fact_24["mask"] == "255.255.255.0"
assert fact_24["and_result"] == "192.168.10.0"
assert SEED == 2408
html_table(
    "Le /24 rendu concret",
    ("IP", "Masque", "AND = réseau", "Broadcast", "Hôtes"),
    ((fact_24["ip"], fact_24["mask"], fact_24["and_result"],
      fact_24["broadcast"], fact_24["usable"]),),
)

## 1. Du préfixe au sous-réseau

Un préfixe **/24** signifie que les 24 premiers bits décrivent le réseau. Son masque est **255.255.255.0** :

- IP : 192.168.10.42 → dernier octet 00101010
- masque : 255.255.255.0 → dernier octet 00000000
- AND : 192.168.10.0 → les bits hôte sont mis à zéro

Pour un préfixe IPv4 classique de /0 à /30 :

- l'adresse réseau identifie le sous-réseau ;
- l'adresse de broadcast vise tous les hôtes du sous-réseau ;
- les adresses entre les deux sont attribuables aux hôtes.

Deux exceptions comptent en exploitation : un **/31** représente couramment les deux extrémités d'un lien point-à-point ; un **/32** désigne une adresse unique, souvent une loopback ou une route d'hôte. Retenir mécaniquement « première et dernière jamais utilisables » serait donc faux.

In [ ]:
prefix_cases = [
    subnet_fact("192.168.10.42/24"),
    subnet_fact("192.168.10.70/26"),
    subnet_fact("10.0.0.4/31"),
    subnet_fact("203.0.113.7/32"),
]

assert prefix_cases[1]["network"] == "192.168.10.64"
assert prefix_cases[1]["broadcast"] == "192.168.10.127"
assert prefix_cases[2]["usable"] == "10.0.0.4 ↔ 10.0.0.5"
assert prefix_cases[3]["usable"] == "203.0.113.7"

html_table(
    "Même IP, frontières différentes",
    ("CIDR", "Masque", "Réseau", "Broadcast / pair", "Plage utile", "Règle"),
    tuple(
        (item["cidr"], item["mask"], item["network"], item["broadcast"],
         item["usable"], item["exception"])
        for item in prefix_cases
    ),
)

## 2. Locale ou distante : laisser la table de routage décider

Une interface configurée en 192.168.10.42/24 crée normalement une route connectée vers 192.168.10.0/24. La machine cherche toutes les routes contenant la destination et choisit le **préfixe le plus long**, donc la route la plus précise.

Dans la table du TP :

- 192.168.10.77 correspond à la route connectée /24 : le prochain saut est la destination elle-même ;
- 198.51.100.8 correspond à une route spécifique /24 via 192.168.10.253, qui bat la route par défaut ;
- 203.0.113.9 ne correspond qu'à 0.0.0.0/0 : la **default gateway** 192.168.10.1 gagne.

Une passerelle IPv4 configurée doit être **on-link** : l'hôte doit pouvoir résoudre son adresse sur un réseau directement connecté. Une gateway 192.168.20.1 est inutilisable depuis 192.168.10.42/24 sans autre route on-link explicite. Après sélection, ARP résout le **prochain saut** : la destination si elle est locale, sinon la passerelle choisie.

In [ ]:
HOST_INTERFACE = ipaddress.ip_interface("192.168.10.42/24")
ROUTES = (
    {"prefix": "192.168.10.0/24", "via": None, "interface": "eth0", "metric": 100, "label": "connectée"},
    {"prefix": "198.51.100.0/24", "via": "192.168.10.253", "interface": "eth0", "metric": 50, "label": "route spécifique"},
    {"prefix": "0.0.0.0/0", "via": "192.168.10.1", "interface": "eth0", "metric": 100, "label": "défaut"},
)

def gateway_is_on_link(gateway, interface=HOST_INTERFACE):
    address = ipaddress.ip_address(gateway)
    network = interface.network
    return (
        address in network
        and address != network.network_address
        and address != network.broadcast_address
    )

def route_lookup(destination, routes=ROUTES):
    target = ipaddress.ip_address(destination)
    candidates = []
    for order, route in enumerate(routes):
        prefix = ipaddress.ip_network(route["prefix"])
        if target in prefix:
            candidates.append((prefix.prefixlen, -route["metric"], -order, route))
    if not candidates:
        return None
    selected = max(candidates)[3]
    next_hop = destination if selected["via"] is None else selected["via"]
    return {
        "destination": destination,
        "route": selected["prefix"],
        "type": selected["label"],
        "interface": selected["interface"],
        "next_hop": next_hop,
        "neighbor_resolution": f"ARP pour {next_hop}",
        "on_link": selected["via"] is None,
    }

routing_cases = tuple(
    route_lookup(destination)
    for destination in ("192.168.10.77", "198.51.100.8", "203.0.113.9")
)
assert routing_cases[0]["next_hop"] == "192.168.10.77"
assert routing_cases[1]["route"] == "198.51.100.0/24"
assert routing_cases[1]["next_hop"] == "192.168.10.253"
assert routing_cases[2]["route"] == "0.0.0.0/0"
assert gateway_is_on_link("192.168.10.1")
assert not gateway_is_on_link("192.168.20.1")

html_table(
    "Préfixe le plus long puis résolution du voisin",
    ("Destination", "Route gagnante", "Décision", "Prochain saut", "Preuve L2"),
    tuple(
        (row["destination"], row["route"], row["type"],
         row["next_hop"], row["neighbor_resolution"])
        for row in routing_cases
    ),
)

## 3. Ce qui change à chaque saut — et ce qui ne change pas

Sans NAT, le paquet conserve ses adresses IP source et destination de bout en bout. En revanche, chaque routeur retire la trame reçue et construit une nouvelle trame pour le lien suivant : les adresses MAC source et destination changent à chaque saut. Le TTL IPv4 diminue aussi. ARP ne cherche donc pas « la MAC du serveur distant » ; il cherche la MAC du prochain saut local.

Ne confondez pas les fonctions :

| Terme | Question traitée |
|---|---|
| **Passerelle** | À quel prochain saut remettre un paquet hors lien ? |
| **Routeur** | Quel équipement transfère entre plusieurs réseaux ? |
| **NAT** | Faut-il traduire une adresse ou un port ? Ce n'est pas implicite dans le routage. |
| **DNS** | Quel nom correspond à quelle adresse ? Il ne décide pas du chemin. |
| **Pare-feu** | Ce trafic est-il autorisé ? Une route existante ne garantit pas l'autorisation. |

Un même équipement peut cumuler ces rôles, mais les confondre empêche de localiser une panne.

In [ ]:
end_to_end = {"src_ip": "192.168.10.42", "dst_ip": "203.0.113.9", "protocol": "TCP"}
hop_frames = (
    {"hop": "LAN A", "src_mac": "02:00:00:00:10:42", "dst_mac": "02:00:00:00:10:01",
     "ttl": 64, "resolved_neighbor": "gateway 192.168.10.1"},
    {"hop": "Transit", "src_mac": "02:00:00:00:aa:01", "dst_mac": "02:00:00:00:bb:01",
     "ttl": 63, "resolved_neighbor": "routeur suivant"},
    {"hop": "LAN B", "src_mac": "02:00:00:00:20:01", "dst_mac": "02:00:00:00:20:09",
     "ttl": 62, "resolved_neighbor": "destination 203.0.113.9"},
)
packet_trace = tuple(
    {
        **frame,
        "src_ip": end_to_end["src_ip"],
        "dst_ip": end_to_end["dst_ip"],
    }
    for frame in hop_frames
)
assert len({row["src_ip"] for row in packet_trace}) == 1
assert len({row["dst_ip"] for row in packet_trace}) == 1
assert len({(row["src_mac"], row["dst_mac"]) for row in packet_trace}) == 3
assert tuple(row["ttl"] for row in packet_trace) == (64, 63, 62)

html_table(
    "Un paquet IP, trois trames",
    ("Lien", "IP source → destination", "MAC source → destination", "TTL", "Voisin résolu"),
    tuple(
        (row["hop"], f"{row['src_ip']} → {row['dst_ip']}",
         f"{row['src_mac']} → {row['dst_mac']}", row["ttl"], row["resolved_neighbor"])
        for row in packet_trace
    ),
)

## 4. Même raisonnement en IPv6, mécanismes différents

Sur un LAN IPv6 courant, le préfixe annoncé est souvent un **/64**. Un Router Advertisement (**RA**) peut annoncer ce préfixe et un routeur par défaut. La route par défaut s'écrit **::/0**. Le prochain saut est fréquemment l'adresse link-local du routeur, par exemple fe80::1.

IPv6 n'a pas de broadcast. Neighbor Discovery (**ND**, messages ICMPv6) remplit le rôle de découverte de voisin ; il utilise notamment le multicast sollicité au lieu d'ARP. La route connectée permet de résoudre directement un voisin du /64, tandis qu'une destination distante part vers le routeur par défaut appris par RA.

Points de vigilance :

- **/64** est la taille standard attendue pour beaucoup de LAN et pour SLAAC, pas une traduction de /24 ;
- on ne déduit pas une « dernière adresse » de passerelle ;
- l'absence de broadcast ne signifie pas absence de trafic multicast ;
- RA annonce des paramètres de lien et une route par défaut, mais n'est ni DNS ni autorisation pare-feu.

In [ ]:
IPV6_INTERFACE = ipaddress.ip_interface("2001:db8:10::42/64")
IPV6_ROUTES = (
    {"prefix": "2001:db8:10::/64", "via": None, "label": "connectée"},
    {"prefix": "::/0", "via": "fe80::1", "label": "défaut appris par RA"},
)

def ipv6_lookup(destination):
    target = ipaddress.ip_address(destination)
    candidates = [
        (ipaddress.ip_network(route["prefix"]).prefixlen, route)
        for route in IPV6_ROUTES
        if target in ipaddress.ip_network(route["prefix"])
    ]
    selected = max(candidates, key=lambda item: item[0])[1]
    next_hop = destination if selected["via"] is None else selected["via"]
    return {
        "destination": destination,
        "route": selected["prefix"],
        "decision": selected["label"],
        "next_hop": next_hop,
        "resolution": f"ND pour {next_hop}",
    }

ipv6_cases = (
    ipv6_lookup("2001:db8:10::77"),
    ipv6_lookup("2001:db8:20::9"),
)
assert IPV6_INTERFACE.network.prefixlen == 64
assert ipv6_cases[0]["route"] == "2001:db8:10::/64"
assert ipv6_cases[1]["route"] == "::/0"
assert ipv6_cases[1]["next_hop"] == "fe80::1"

html_table(
    "IPv6 : route puis ND, jamais ARP",
    ("Destination", "Route", "Décision", "Prochain saut", "Résolution"),
    tuple(
        (row["destination"], row["route"], row["decision"],
         row["next_hop"], row["resolution"])
        for row in ipv6_cases
    ),
)

## 5. Diagnostiquer par invariant, pas par intuition

Trois incidents proches en apparence produisent des preuves différentes :

1. **Mauvais masque** : configuré en /16 au lieu du /24 réel, l'hôte croit 192.168.99.20 local et répète des ARP sans réponse. La gateway n'est même pas sollicitée.
2. **Gateway hors lien** : 192.168.20.1 n'appartient pas au lien 192.168.10.0/24. Le prochain saut ne peut pas être résolu directement.
3. **Route spécifique erronée** : 10.20.0.0/16 via 192.168.10.253 bat le défaut. Seule cette destination échoue ; changer le DNS ne corrige rien.

Ordre de preuve conseillé : configuration adresse/préfixe → routes candidates → route gagnante → prochain saut on-link → résolution ARP/ND → filtre éventuel. Un ping seul ne révèle pas l'étape fautive.

In [ ]:
incident_report = (
    {
        "incident": "Mauvais masque /16",
        "observation": "ARP répétés pour 192.168.99.20",
        "invariant_broken": "frontière locale réelle = /24",
        "proof": "route connectée calculée 192.168.0.0/16",
        "correction": "rétablir 192.168.10.42/24",
    },
    {
        "incident": "Gateway hors lien",
        "observation": "aucune MAC pour 192.168.20.1",
        "invariant_broken": "prochain saut directement joignable",
        "proof": "192.168.20.1 ∉ 192.168.10.0/24",
        "correction": "gateway 192.168.10.1 ou route on-link valide",
    },
    {
        "incident": "Route spécifique erronée",
        "observation": "10.20.0.0/16 échoue, Internet fonctionne",
        "invariant_broken": "route la plus précise pointe vers le bon routeur",
        "proof": "/16 via .253 gagne sur /0 via .1",
        "correction": "corriger ou retirer la route /16",
    },
)

assert len(incident_report) == 3
assert "ARP" in incident_report[0]["observation"]
assert "∉" in incident_report[1]["proof"]

html_table(
    "Matrice d'incidents : symptôme → preuve → correction",
    ("Incident", "Observation", "Invariant cassé", "Preuve minimale", "Correction"),
    tuple(
        (item["incident"], item["observation"], item["invariant_broken"],
         item["proof"], item["correction"])
        for item in incident_report
    ),
)

In [ ]:
def network_svg():
    rows = [
        ("192.168.10.77", "192.168.10.0/24", "ARP destination", "#34d399"),
        ("198.51.100.8", "198.51.100.0/24", "ARP 192.168.10.253", "#fbbf24"),
        ("203.0.113.9", "0.0.0.0/0", "ARP 192.168.10.1", "#60a5fa"),
    ]
    cards = []
    for index, (destination, route, action, color) in enumerate(rows):
        x = 30 + index * 280
        cards.append(
            f"<g transform='translate({x},390)'>"
            f"<rect width='250' height='92' rx='14' fill='#111c33' stroke='{color}' stroke-width='2'/>"
            f"<text x='16' y='27' fill='{color}' font-size='14' font-weight='700'>{destination}</text>"
            f"<text x='16' y='51' fill='#e2e8f0' font-size='12'>route {route}</text>"
            f"<text x='16' y='73' fill='#94a3b8' font-size='12'>{action}</text></g>"
        )
    return f"""<svg xmlns='http://www.w3.org/2000/svg' width='900' height='520' viewBox='0 0 900 520' role='img' aria-label='Décision de sous-réseau et de passerelle'>
<defs>
  <linearGradient id='bg' x1='0' x2='1'><stop stop-color='#07111f'/><stop offset='1' stop-color='#13213d'/></linearGradient>
  <marker id='arrow' markerWidth='8' markerHeight='8' refX='7' refY='4' orient='auto'><path d='M0,0 L8,4 L0,8 z' fill='#7dd3fc'/></marker>
</defs>
<rect width='900' height='520' rx='20' fill='url(#bg)'/>
<text x='30' y='42' fill='#f8fafc' font-size='23' font-family='system-ui' font-weight='700'>Destination → route la plus précise → prochain saut → voisin L2</text>
<text x='30' y='69' fill='#94a3b8' font-size='13' font-family='system-ui'>192.168.10.42/24 · table synthétique · aucune émission réseau</text>

<g font-family='system-ui'>
  <rect x='35' y='125' width='180' height='105' rx='18' fill='#10223c' stroke='#34d399' stroke-width='2'/>
  <text x='55' y='157' fill='#34d399' font-size='16' font-weight='700'>HÔTE</text>
  <text x='55' y='184' fill='#e2e8f0' font-size='13'>192.168.10.42/24</text>
  <text x='55' y='207' fill='#94a3b8' font-size='12'>MAC 02:…:10:42</text>

  <path d='M215 178 H315' stroke='#7dd3fc' stroke-width='3' marker-end='url(#arrow)'/>
  <text x='230' y='162' fill='#7dd3fc' font-size='11'>LAN / ARP</text>

  <rect x='315' y='105' width='245' height='145' rx='18' fill='#182640' stroke='#7dd3fc' stroke-width='2'/>
  <text x='338' y='137' fill='#7dd3fc' font-size='16' font-weight='700'>TABLE DE ROUTAGE</text>
  <text x='338' y='166' fill='#e2e8f0' font-size='12'>192.168.10.0/24 · connecté</text>
  <text x='338' y='191' fill='#fbbf24' font-size='12'>198.51.100.0/24 · via .253</text>
  <text x='338' y='216' fill='#60a5fa' font-size='12'>0.0.0.0/0 · via .1</text>

  <path d='M560 178 H655' stroke='#7dd3fc' stroke-width='3' marker-end='url(#arrow)'/>
  <text x='572' y='162' fill='#7dd3fc' font-size='11'>trame par saut</text>

  <rect x='655' y='125' width='205' height='105' rx='18' fill='#10223c' stroke='#60a5fa' stroke-width='2'/>
  <text x='675' y='157' fill='#60a5fa' font-size='16' font-weight='700'>ROUTEUR</text>
  <text x='675' y='184' fill='#e2e8f0' font-size='13'>gateway 192.168.10.1</text>
  <text x='675' y='207' fill='#94a3b8' font-size='12'>NAT ≠ routage</text>

  <path d='M125 230 V296 H758 V230' stroke='#475569' stroke-width='2' stroke-dasharray='7 6'/>
  <text x='288' y='320' fill='#cbd5e1' font-size='13'>IP 192.168.10.42 → 203.0.113.9 reste inchangée sans NAT</text>
  <text x='318' y='344' fill='#94a3b8' font-size='12'>les MAC changent ; le TTL diminue à chaque routeur</text>
  {''.join(cards)}
</g>
</svg>"""

topology_svg = network_svg()
assert topology_svg.startswith("<svg")
assert "0.0.0.0/0" in topology_svg
RichHTML(topology_svg)

## 6. Cas de transfert, exports et limites

Cas inédit : un poste 172.16.4.18/23 possède une route connectée 172.16.4.0/23, une route 10.40.0.0/16 via 172.16.5.254 et une route par défaut via 172.16.4.1. Pour 10.40.7.9, prédisez avant d'exécuter :

1. la route gagnante ;
2. le prochain saut ;
3. l'adresse recherchée par ARP ;
4. les adresses IP qui resteront dans le paquet sans NAT.

**Autorisation :** le plan exporté est inerte. Il décrit des vérifications à réaliser uniquement sur un réseau ou un laboratoire explicitement autorisé ; il ne lance ni capture, ni scan, ni reconfiguration.

**Limites :** le modèle omet ECMP, policy routing, VRF, proxy ARP, tunnels, VLAN, états pare-feu, ICMP Redirect, fragmentation et variations propres aux systèmes. Les adresses sont réservées à la documentation. Une vraie preuve exige la configuration, la table de routage, la table de voisins et une capture autorisée du système étudié.

In [ ]:
TRANSFER_INTERFACE = ipaddress.ip_interface("172.16.4.18/23")
TRANSFER_ROUTES = (
    {"prefix": "172.16.4.0/23", "via": None, "interface": "eth0", "metric": 100, "label": "connectée"},
    {"prefix": "10.40.0.0/16", "via": "172.16.5.254", "interface": "eth0", "metric": 20, "label": "route spécifique"},
    {"prefix": "0.0.0.0/0", "via": "172.16.4.1", "interface": "eth0", "metric": 100, "label": "défaut"},
)
transfer = route_lookup("10.40.7.9", TRANSFER_ROUTES)
assert transfer["route"] == "10.40.0.0/16"
assert transfer["next_hop"] == "172.16.5.254"
assert ipaddress.ip_address(transfer["next_hop"]) in TRANSFER_INTERFACE.network

routing_plan = {
    "execution_allowed": False,
    "synthetic": True,
    "seed": SEED,
    "host": str(TRANSFER_INTERFACE),
    "destination": transfer["destination"],
    "selected_route": transfer["route"],
    "next_hop": transfer["next_hop"],
    "neighbor_action": transfer["neighbor_resolution"],
    "ip_invariant_without_nat": {
        "source": str(TRANSFER_INTERFACE.ip),
        "destination": transfer["destination"],
    },
    "authorized_evidence_plan": [
        "inspect address and prefix",
        "inspect routing table",
        "inspect ARP or ND neighbor state",
        "capture only with explicit authorization",
    ],
    "incidents": incident_report,
}
report_bytes = json.dumps(
    routing_plan, ensure_ascii=False, indent=2, sort_keys=True
).encode("utf-8")
report_digest = hashlib.sha256(report_bytes).hexdigest()
report_uri = "data:application/json;base64," + base64.b64encode(report_bytes).decode("ascii")

csv_lines = ["destination,route,next_hop,resolution"]
for row in routing_cases:
    csv_lines.append(
        ",".join((row["destination"], row["route"], row["next_hop"], row["neighbor_resolution"]))
    )
csv_bytes = ("\n".join(csv_lines) + "\n").encode("utf-8")
csv_uri = "data:text/csv;base64," + base64.b64encode(csv_bytes).decode("ascii")

assert routing_plan["execution_allowed"] is False
assert report_uri.startswith("data:application/json;base64,")
assert csv_uri.startswith("data:text/csv;base64,")

RichHTML(
    "<div style='font:14px system-ui;background:#ecfeff;border:1px solid #06b6d4;"
    "border-radius:12px;padding:16px;color:#164e63'>"
    "<strong>Cas de transfert validé :</strong> 10.40.0.0/16 gagne ; ARP vise 172.16.5.254."
    f"<br><span style='font-family:monospace'>SHA-256 {report_digest[:20]}…</span>"
    f"<br><a download='tp8-routing-report.json' href='{report_uri}'>Télécharger le rapport JSON</a>"
    f" · <a download='tp8-routing-decisions.csv' href='{csv_uri}'>Télécharger la matrice CSV</a>"
    "<br><small>Exports synthétiques et plan inerte : aucune action réseau.</small></div>"
)

## 7. Auto-évaluation

Sans relire, complétez ces phrases :

1. /24 correspond au masque ______ et l'adresse réseau résulte de l'opération ______.
2. Pour une destination locale, ARP cherche ______ ; pour une destination distante, ARP cherche ______.
3. Entre plusieurs routes correspondantes, la route au préfixe ______ gagne.
4. Sans NAT, les adresses ______ restent stables, mais les adresses ______ changent par saut.
5. En IPv6, ______ remplace ARP, la route par défaut est ______ et il n'existe pas de ______.
6. Une réponse DNS correcte ne prouve ni qu'une ______ existe, ni que le ______ autorise le flux.

Réponses attendues : 255.255.255.0 ; AND ; la destination ; le prochain saut ; le plus long ; IP ; MAC ; ND ; ::/0 ; broadcast ; route ; pare-feu.

Critère de réussite : vous pouvez justifier chaque décision avec une entrée de table, un préfixe et un voisin à résoudre, sans réduire « tout ce qui est distant » à une passerelle magique.